<a href="https://colab.research.google.com/github/wjohn564/CS6271-2025-6---Final-Project/blob/main/20245424_EA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
<img src="https://www.ul.ie/themes/custom/ul/logo.jpg" />
</div>

#**MSc in Artificial Intelligence and Machine Learning**
##CS6271 - Evolutionary Algorithms and Humanoid Robotics 2025
### Kaggle Competition


Module Leader: Conor Ryan

Developer: John Walsh

Predict whether a person will earn more than 50k or less. This is a modified version of the adult dataset

In [1]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Dataset

**Class:**

output: 1, 0.


**Listing of features:**

- 'age'
- 'workclass'
- 'fnlwgt'
- 'education'
- 'education-num'
- 'marital-status'
- 'occupation'
- 'relationship'
- 'race'
- 'sex'
- 'capital-gain'
- 'capital-loss'
- 'hours-per-week'
- 'native-country'

In [2]:
# Suppressing Warnings:
import warnings
warnings.filterwarnings("ignore")

In [3]:
## mount your Google drive
# 1) run this cell
# 2) sign in
# 3) verify your drive is mounted

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Grape Setup for Grammatical Evolution

In [4]:
import os
# Get the library from our BDS research Group
# copy the path from your drive
PATH = '/content/drive/MyDrive/grape/'

# check if 'grape' already exists
if os.path.exists(PATH):
    print('grape directory already exists')
else:
    %cd /content/drive/MyDrive/
    !git clone https://github.com/bdsul/grape.git
    print('Cloning grape in your Drive')

# change directory to 'grape'
%cd /content/drive/MyDrive/grape/

grape directory already exists
/content/drive/MyDrive/grape


## Load the training dataset, visualise some stats

In [5]:
import zipfile

zip_file_path = '/content/cs-6271-2025-6-final-project.zip'
extraction_path = '/content/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Extracted contents of {zip_file_path} to {extraction_path}")
print("Files in extraction path:")
print(os.listdir(extraction_path))

Extracted contents of /content/cs-6271-2025-6-final-project.zip to /content/
Files in extraction path:
['.config', 'test.csv', 'sample_submission_with_id.csv', 'drive', 'cs-6271-2025-6-final-project.zip', 'train.csv', 'sample_data']


In [6]:
# Load the Training Set
train_file = '/content/train.csv'
df_train = pd.read_csv(train_file)

# Show the first 5 rows
df_train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,37,Private,193106,Bachelors,13,Never-married,Sales,Not-in-family,White,Female,0,0,30,United-States,0
1,56,Self-emp-inc,216636,12th,8,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,1651,40,United-States,0
2,53,Private,126977,HS-grad,9,Separated,Craft-repair,Not-in-family,White,Male,0,0,35,United-States,0
3,72,Private,205343,11th,7,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
4,46,State-gov,106705,Masters,14,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,38,United-States,0


In [7]:
# Visualise some helpful details
df_train.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,income
count,39073.000000,3.907300e+04,39073.000000,39073.000000,39073.000000,39073.000000,39073.000000
mean,38.643488,1.899922e+05,10.069844,1038.040540,86.807949,40.476877,0.239270
std,13.685634,1.054768e+05,2.574387,7204.953114,401.276773,12.401251,0.426643
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,1.000000,0.000000
25%,28.000000,1.177670e+05,9.000000,0.000000,0.000000,40.000000,0.000000
50%,37.000000,1.786150e+05,10.000000,0.000000,0.000000,40.000000,0.000000
75%,48.000000,2.383290e+05,12.000000,0.000000,0.000000,45.000000,0.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000,1.000000


In [8]:
# More stats to view
df_train.describe(include='object')

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
count,38304,39073,39073,38302,39073,39073,39073,38851
unique,9,16,7,15,6,5,2,42
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States
freq,27141,12582,17951,4915,15822,33414,26170,35059


# Load the Test dataset (**NO LABELS**), visualise some stats and create the target variable

In [9]:
# Load the Test Set

train_file = '/content/test.csv'
df_test = pd.read_csv(train_file)

# Show the first 5 rows
df_test.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,30,Private,378009,HS-grad,9,Never-married,Machine-op-inspct,Own-child,White,Male,0,0,40,United-States
1,54,State-gov,55861,Assoc-acdm,12,Divorced,Adm-clerical,Not-in-family,White,Male,0,0,39,United-States
2,21,?,204226,Some-college,10,Never-married,?,Unmarried,White,Female,0,0,35,United-States
3,35,Private,306678,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,2885,0,40,United-States
4,42,Local-gov,121012,Some-college,10,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States


In [10]:
# Visualise some helpful details
df_test.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,9769.000000,9.769000e+03,9769.000000,9769.000000,9769.000000,9769.000000
mean,38.643976,1.883518e+05,10.111066,1243.163374,90.279558,40.204422
std,13.810263,1.061065e+05,2.557138,8365.976609,409.851357,12.350369
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,27.000000,1.164930e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.767320e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.345420e+05,13.000000,0.000000,0.000000,45.000000
max,90.000000,1.366120e+06,16.000000,99999.000000,3770.000000,99.000000


In [11]:
# More stats to view
df_test.describe(include='object')

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
count,9575,9769,9769,9574,9769,9769,9769,9717
unique,9,16,7,15,6,5,2,41
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States
freq,6765,3202,4428,1257,3894,8348,6480,8773


# Data Preprocessing


Some Helper Functions

In [12]:
from sklearn.preprocessing import LabelEncoder

def add_label_encoder_to_columns(df, categorical_cols):
  """
  This function will be used on the training data to map integars to the
  Categorical columns. It does this by fitting a label encoder onto each categorical
  column.

  The function returns a dictionary: column_name: encoder
  """
  encoders = {}

  for col in categorical_cols:
    # instantiate the LabelEncoder class
    le = LabelEncoder()
    # give NaN type values 'Unknown'
    types = df[col].fillna('Unknown').astype(str)
    # create mappings for categories to numeric values
    le.fit(types)
    # add the encoder to the dictionary
    encoders[col] = le
  return encoders


def apply_label_encoder(df, encoders, categorical_cols, numerical_cols):
  """
  This function will apply any previously defined encoder to the
  supplied dataframe.

  returns a fully numerical dataframe
  """
  # Create a copy to work with
  df = df.copy()

  # Encode the categorical columns
  for col in categorical_cols:
    le = encoders[col]
    types = df[col].fillna('Unknown').astype(str)

    # quick check to make sure 'Unknown' is a new encoder class
    if 'Unknown' not in le.classes_:
      le.classes_ = np.append(le.classes_, 'Unknown')

    # check for categories not picked up from initial mapping
    # assign 'unknown' to those types
    mask_unknown = ~np.isin(types, le.classes_)
    if mask_unknown.any():
      types[mask_unknown] = 'Unknown'

    # finally apply the encoding
    df[col] = le.transform(types)


  # convert numeric cols to floats
  for col in numerical_cols:
    df[col] = df[col].astype(float)

  # concatinate them together
  feature_cols = numerical_cols + categorical_cols
  return df[feature_cols]



Applying Encoders

In [13]:
# List the categorical cols
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation',
                    'relationship', 'race', 'sex', 'native-country'
                    ]

# List the numerical cols
numerical_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain',
                  'capital-loss', 'hours-per-week'
                  ]

# remove target variable from training data and create copies of datasets to
# work with
y = df_train['income']
X_train = df_train.drop(columns=['income'])
X_test = df_test.copy()

# Use the function above to fit an encoder on the training set
encoders = add_label_encoder_to_columns(X_train, categorical_cols)

# Apply the encoder to the training and test sets
X_train = apply_label_encoder(X_train, encoders, categorical_cols, numerical_cols)
X_test = apply_label_encoder(X_test, encoders, categorical_cols, numerical_cols)

print(f"Encoded training set shape: {X_train.shape}")
print(f"Encoded test set shape: {X_test.shape}")

Encoded training set shape: (39073, 14)
Encoded test set shape: (9769, 14)


Final touch up for grape

In [14]:
# put in numpy array format
Y_train = y.to_numpy()

# Applied the transpose as per the comment given
X_train = X_train.to_numpy().T
X_test = X_test.to_numpy().T

### NB***** make sure this cell doesnt run again or it will apply another transpose
# and then it will not be correct
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape} ")
print(f"Y_train: {Y_train.shape}")

X_train: (14, 39073)
X_test: (14, 9769) 
Y_train: (39073,)


## GRAPE

<div>
<img src="https://drive.google.com/uc?export=view&id=1hw43Oi3lGTCkspQ0ged2bZB8q2EpcPhz" width="150"/>
</div>

GRammatical Algorithms in Python for Evolution (GRAPE)

In [15]:
!pip install deap

import grape
import algorithms

from os import path
from deap import creator, base, tools
import random
import csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 5.9 MB/s eta 0:00:00


Import the functions to be used with the grammar from [functions.py](https://github.com/UL-BDS/grape/blob/main/functions.py) int the GRAPE repository

In [16]:
from functions import and_, or_, not_, less_than_or_equal, greater_than_or_equal

Create a bnf file with the Grammar

Note: The grammar style I am going with is inspired by Conor Ryans github repo: [HERE](https://github.com/bdsul/grape/blob/main/grammars/parity4.bnf)


In [17]:
# This cell is to create Grammar and save it as a bnf file

# The Feature indices:
# 0: age
# 1: fnlwgt
# 2: education-num
# 3: capital-gain
# 4: capital-loss
# 5: hours-per-week
# 6: workclass
# 7: education (this one is the categorical feature)
# 8: marital-status
# 9: occupation
# 10: relationship
# 11: race
# 12: sex
# 13: native-country

adult_grammar = r"""
<log_op> ::= <conditional>
           | and_(<log_op>, <log_op>)
           | or_(<log_op>, <log_op>)
           | not_(<log_op>)

<conditional> ::= greater_than_or_equal(<feature>, <constant>)
                | less_than_or_equal(<feature>, <constant>)

<feature> ::= x[0]  | x[1]  | x[2]  | x[3]  | x[4]  | x[5]
            | x[6]  | x[7]  | x[8]  | x[9]  | x[10] | x[11]
            | x[12] | x[13]

<constant> ::= 0 | 5 | 10 | 15 | 20 | 25 | 30 | 35 | 40 | 45
             | 50 | 55 | 60 | 65 | 70 | 75 | 80
"""

with open("/content/adult_income.bnf", "w") as f:
    f.write(adult_grammar)

In [18]:
GRAMMAR_FILE = "/content/adult_income.bnf"
# Open and display Grammar
f = open( GRAMMAR_FILE, "r")
print(f.read())
f.close()


<log_op> ::= <conditional>
           | and_(<log_op>, <log_op>)
           | or_(<log_op>, <log_op>)
           | not_(<log_op>)

<conditional> ::= greater_than_or_equal(<feature>, <constant>)
                | less_than_or_equal(<feature>, <constant>)

<feature> ::= x[0]  | x[1]  | x[2]  | x[3]  | x[4]  | x[5]
            | x[6]  | x[7]  | x[8]  | x[9]  | x[10] | x[11]
            | x[12] | x[13]

<constant> ::= 0 | 5 | 10 | 15 | 20 | 25 | 30 | 35 | 40 | 45
             | 50 | 55 | 60 | 65 | 70 | 75 | 80



The following cell puts the grammar on the class Grammar.

In [19]:
BNF_GRAMMAR = grape.Grammar(GRAMMAR_FILE)

I am using the supplied fitness function which tracks the percentage of outputs wrongly predicted.

In [20]:
def fitness_eval(individual, points):
    """
    Fitness Function
    """

    x = points[0]
    Y = points[1]

    if individual.invalid == True:
        return np.nan,

    # Evaluate the expression
    try:
        pred = eval(individual.phenotype)
    except (FloatingPointError, ZeroDivisionError, OverflowError,
            MemoryError):
        return np.nan,
    assert np.isrealobj(pred)

    compare = np.equal(Y,pred)
    fitness = 1 - np.mean(compare)

    return fitness,

Grammatical Evolution parameters.

In [21]:
POPULATION_SIZE = 400
MAX_GENERATIONS = 60
P_CROSSOVER = 0.9
P_MUTATION = 0.1
ELITE_SIZE = 1
HALL_OF_FAME_SIZE = 1

TOURNAMENT_SIZE = 4
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# How codons are consumed and stored
CODON_CONSUMPTION = 'lazy'
GENOME_REPRESENTATION = 'list'
CODON_SIZE = 255
MAX_GENOME_LENGTH = 100

# Random initialisation
MIN_INIT_GENOME_LENGTH = 30
MAX_INIT_GENOME_LENGTH = 80

# Controls for tree depth
MAX_INIT_TREE_DEPTH = 8
MAX_TREE_DEPTH      = 12
MAX_WRAPS           = 0

REPORT_ITEMS = ['gen', 'invalid', 'avg', 'std', 'min', 'max',
                'best_ind_length', 'avg_length',
                'best_ind_nodes', 'avg_nodes',
                'best_ind_depth', 'avg_depth',
                'avg_used_codons', 'best_ind_used_codons',
                'structural_diversity', 'fitness_diversity',
                'selection_time', 'generation_time']

In [22]:
toolbox = base.Toolbox()

# define a single objective, minimising fitness strategy:
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

creator.create('Individual', grape.Individual, fitness=creator.FitnessMin)

toolbox.register("populationCreator", grape.random_initialisation, creator.Individual)

toolbox.register("evaluate", fitness_eval)

# Tournament selection:
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

# Single-point crossover:
toolbox.register("mate", grape.crossover_onepoint)

# Flip-int mutation:
toolbox.register("mutate", grape.mutation_int_flip_per_codon)

In [23]:
# create initial population (generation 0):
population = toolbox.populationCreator(
    pop_size=POPULATION_SIZE,
    bnf_grammar=BNF_GRAMMAR,
    min_init_genome_length=MIN_INIT_GENOME_LENGTH,
    max_init_genome_length=MAX_INIT_GENOME_LENGTH,
    max_init_depth=MAX_INIT_TREE_DEPTH,
    codon_size=CODON_SIZE,
    codon_consumption=CODON_CONSUMPTION,
    genome_representation=GENOME_REPRESENTATION,
)

# define the hall-of-fame object:
hof = tools.HallOfFame(HALL_OF_FAME_SIZE)

# prepare the statistics object:
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.nanmean)
stats.register("std", np.nanstd)
stats.register("min", np.nanmin)
stats.register("max", np.nanmax)

Run GE

In [24]:
population, logbook = algorithms.ge_eaSimpleWithElitism(population, toolbox, cxpb=P_CROSSOVER, mutpb=P_MUTATION,
                                              ngen=MAX_GENERATIONS, elite_size=ELITE_SIZE,
                                              bnf_grammar=BNF_GRAMMAR,
                                              codon_size=CODON_SIZE,
                                              max_tree_depth=MAX_TREE_DEPTH,
                                              max_genome_length=MAX_GENOME_LENGTH,
                                              points_train=[X_train, Y_train],
                                              codon_consumption=CODON_CONSUMPTION,
                                              report_items=REPORT_ITEMS,
                                              genome_representation=GENOME_REPRESENTATION,
                                              stats=stats, halloffame=hof, verbose=False)

gen = 0 , Best fitness = (np.float64(0.2200752437744734),)
gen = 1 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 0
gen = 2 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 0
gen = 3 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 4
gen = 4 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 1
gen = 5 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 0
gen = 6 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 2
gen = 7 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 0
gen = 8 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 1
gen = 9 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 1
gen = 10 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 1
gen = 11 , Best fitness = (np.float64(0.2200752437744734),) , Number of invalids = 0
gen = 12 , Best

show the best individual as an expression

In [25]:
# Best individual
import textwrap
best = hof.items[0].phenotype
print("Best individual: \n","\n".join(textwrap.wrap(best,80)))
print("\nTraining Fitness: ", hof.items[0].fitness.values[0])

Best individual: 
 and_(greater_than_or_equal(x[2], 10), less_than_or_equal(x[10], 0))

Training Fitness:  0.19739973895017016


Define a function to predict values, without comparing to expected outputs.

In [26]:
def predict(individual, X):
    x = X

    if individual.invalid == True:
        return np.NaN,

    # Evaluate the expression
    try:
        pred = eval(individual.phenotype)
    except (FloatingPointError, ZeroDivisionError, OverflowError,
            MemoryError):
        return np.NaN,
    assert np.isrealobj(pred)

    return pred

# Predict the classes of the test set.


In [27]:
y_pred = predict(hof.items[0], X_test)
print("Predicted classes of the test set: ", y_pred)

Predicted classes of the test set:  [False False False ... False False  True]
